# 02 — Analysis: alignment gaps, crossed constituencies, thresholds, E8
Runs on CPU (no GPU needed). Input: `guard_scores_full.csv` from notebook 01.
Produces every number and figure the paper's \todo{} slots require, plus LaTeX rows for Table 1.

**Pre-registration**: commit the analysis choices in the next cell to the repo BEFORE looking at full-run results.

In [ ]:
import numpy as np, pandas as pd, itertools, math, json
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt

# ---------- PRE-REGISTERED CHOICES (edit only before seeing results) ----------
PRIMARY_CODING   = "unsafe"        # Q_overall == 'Yes'; sensitivity: 'unsafe_sens'
GROUP_AXES       = ["rater_race", "rater_gender", "rater_age", "rater_education"]
PRIMARY_AXIS     = "rater_race"
N_BOOT           = 2000
MIN_RATERS       = 10              # min raters per group per item for that group's majority
THRESH_RULE      = "base_rate_matched"   # flag top q of scores, q = pooled human unsafe rate
SEED             = 20260717
rng = np.random.default_rng(SEED)

DICES_URL = ("https://raw.githubusercontent.com/google-research-datasets/"
             "dices-dataset/main/350/diverse_safety_adversarial_dialog_350.csv")
df = pd.read_csv(DICES_URL)
df["unsafe"]      = (df.Q_overall == "Yes").astype(int)
df["unsafe_sens"] = (df.Q_overall.isin(["Yes","Unsure"])).astype(int)
scores = pd.read_csv("guard_scores_full.csv")
GUARD_COLS = [c for c in scores.columns if c.startswith("p_")]
print("guards:", GUARD_COLS)

## Human-side structures (both codings)

In [ ]:
def build(coding):
    lab = df.groupby("item_id")[coding].mean().rename("pooled_rate")
    H = -(np.clip(lab,1e-9,1-1e-9)*np.log2(np.clip(lab,1e-9,1-1e-9))
          + np.clip(1-lab,1e-9,1-1e-9)*np.log2(np.clip(1-lab,1e-9,1-1e-9)))
    maj = {}
    for ax in GROUP_AXES:
        counts = df.groupby(["item_id",ax])[coding].count().unstack()
        means  = df.groupby(["item_id",ax])[coding].mean().unstack()
        maj[ax] = (means > .5).astype(float).where(counts >= MIN_RATERS)
    return lab, H.rename("entropy"), maj

pooled, entropy, MAJ = build(PRIMARY_CODING)
race = MAJ[PRIMARY_AXIS]
contested = race.index[race.nunique(axis=1) > 1]
print(f"contested (opposing race majorities): {len(contested)} items")
print("pooled unsafe rate: %.3f" % pooled.mean())

## Core metrics per guard: AUC & κ by group — full set and contested subset

In [ ]:
def guard_table(col, maj, subset=None):
    s = scores.set_index("item_id")[col]
    idx = maj.index if subset is None else maj.index.intersection(subset)
    s = s.loc[idx]; ok = s.notna()
    thr = np.nanquantile(s, 1 - pooled.mean()) if THRESH_RULE=="base_rate_matched" else 0.5
    v = (s >= thr).astype(float)
    rows = {}
    for grp in maj.columns:
        gm = maj[grp].loc[idx]
        m = ok & gm.notna()
        if m.sum() < 20 or gm[m].nunique() < 2: rows[grp] = (np.nan, np.nan, int(m.sum())); continue
        rows[grp] = (roc_auc_score(gm[m], s[m]),
                     cohen_kappa_score(gm[m].astype(int), v[m].astype(int)),
                     int(m.sum()))
    return rows, float(thr)

def gaps(rows):
    aucs = [r[0] for r in rows.values() if not math.isnan(r[0])]
    kaps = [r[1] for r in rows.values() if not math.isnan(r[1])]
    return (max(aucs)-min(aucs) if aucs else np.nan,
            max(kaps)-min(kaps) if kaps else np.nan)

report = {}
for col in GUARD_COLS:
    full_rows, thr = guard_table(col, race)
    sub_rows,  _   = guard_table(col, race, subset=contested)
    ag_f, kg_f = gaps(full_rows); ag_s, kg_s = gaps(sub_rows)
    report[col] = dict(threshold=round(thr,4),
        full={g[:12]: (round(r[0],3), round(r[1],3), r[2]) for g,r in full_rows.items()},
        auc_gap_full=round(ag_f,3), kappa_gap_full=round(kg_f,3),
        auc_gap_contested=round(ag_s,3), kappa_gap_contested=round(kg_s,3))
print(json.dumps(report, indent=1))

## Bootstrap CIs (2000 resamples, item-level)

In [ ]:
def boot_gaps(col, maj):
    s = scores.set_index("item_id")[col].loc[maj.index]
    items = maj.index[s.notna()].to_numpy()
    out = []
    for _ in range(N_BOOT):
        bi = rng.choice(items, len(items), replace=True)
        sb = s.loc[bi].to_numpy()
        thr = np.nanquantile(sb, 1 - pooled.mean()); vb = (sb >= thr)
        aucs, kaps = [], []
        for grp in maj.columns:
            gm = maj[grp].loc[bi].to_numpy()
            m = ~np.isnan(gm)
            if m.sum() < 20 or len(set(gm[m])) < 2: continue
            aucs.append(roc_auc_score(gm[m], sb[m]))
            kaps.append(cohen_kappa_score(gm[m].astype(int), vb[m].astype(int)))
        out.append((max(aucs)-min(aucs) if aucs else np.nan,
                    max(kaps)-min(kaps) if kaps else np.nan))
    a = np.nanpercentile([o[0] for o in out],[2.5,97.5])
    k = np.nanpercentile([o[1] for o in out],[2.5,97.5])
    return np.round(a,3).tolist(), np.round(k,3).tolist()

for col in GUARD_COLS:
    a_ci, k_ci = boot_gaps(col, race)
    report[col]["auc_gap_ci95"] = a_ci; report[col]["kappa_gap_ci95"] = k_ci
    print(col, "AUC gap CI", a_ci, "| kappa gap CI", k_ci)

## Crossed constituencies — Kendall's τ between guards' group orderings
The headline test. Low/negative τ between guard pairs = orderings cross = guard choice is constituency choice.

In [ ]:
order = {}
for col in GUARD_COLS:
    aucs = {g: report[col]["full"][g[:12]][0] for g in race.columns}
    order[col] = pd.Series(aucs).rank()
taus = {}
for a, b in itertools.combinations(GUARD_COLS, 2):
    t, p = kendalltau(order[a], order[b])
    taus[f"{a} vs {b}"] = (round(t,2), round(p,3))
    print(f"{a:22s} vs {b:22s}: tau={t:+.2f} p={p:.3f}")
print("\nBest-tracked group per guard:",
      {c: pd.Series({g: report[c]['full'][g[:12]][0] for g in race.columns}).idxmax()[:12] for c in GUARD_COLS})

## Disagreement sensitivity + mixed-effects model (GEE, item-clustered)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

for col in GUARD_COLS:
    s = scores.set_index("item_id")[col].loc[race.index]
    ok = s.notna()
    conf = (s[ok] - s[ok].median()).abs()
    rho, p = spearmanr(conf, entropy.loc[s.index[ok]])
    print(f"{col}: conf~entropy rho={rho:+.3f} p={p:.4f}")

# GEE: agreement(guard, group majority) ~ group, exchangeable correlation within item
long = []
for col in GUARD_COLS:
    s = scores.set_index("item_id")[col].loc[race.index]
    thr = np.nanquantile(s, 1 - pooled.mean()); v = (s >= thr).astype(float)
    for grp in race.columns:
        gm = race[grp]
        m = s.notna() & gm.notna()
        for iid in race.index[m]:
            long.append(dict(guard=col, group=grp[:12], item=iid,
                             agree=int(v.loc[iid] == gm.loc[iid])))
L = pd.DataFrame(long)
for col in GUARD_COLS:
    sub = L[L.guard == col]
    gee = smf.gee("agree ~ C(group)", groups="item", data=sub,
                  family=sm.families.Binomial(),
                  cov_struct=sm.cov_struct.Exchangeable()).fit()
    print(f"\n==== {col} ====\n", gee.summary().tables[1])

## Fig. 4 — threshold-sweep constituency map

In [ ]:
fig, axes = plt.subplots(1, len(GUARD_COLS), figsize=(4*len(GUARD_COLS), 3), sharey=True)
if len(GUARD_COLS) == 1: axes = [axes]
for ax_, col in zip(axes, GUARD_COLS):
    s = scores.set_index("item_id")[col].loc[race.index]
    ok = s.notna()
    ts = np.quantile(s[ok], np.linspace(.05, .95, 60))
    best = []
    for t in ts:
        v = (s[ok] >= t).astype(int)
        kk = {}
        for grp in race.columns:
            gm = race[grp].loc[s.index[ok]]
            m = gm.notna()
            if gm[m].nunique() > 1 and v[m].nunique() > 1:
                kk[grp[:12]] = cohen_kappa_score(gm[m].astype(int), v[m])
        best.append(max(kk, key=kk.get) if kk else None)
    groups = sorted(set(b for b in best if b))
    cmap = {g: i for i, g in enumerate(groups)}
    ax_.scatter(ts, [cmap.get(b, -1) for b in best], s=8)
    ax_.set_yticks(range(len(groups))); ax_.set_yticklabels(groups, fontsize=7)
    ax_.set_title(col.replace("p_",""), fontsize=9); ax_.set_xlabel("threshold")
plt.tight_layout(); plt.savefig("fig4_constituency.pdf"); plt.show()
print("If the best-served group changes along x for any guard, Contribution 3 lands.")

## E8 — advice-harm stratification (exploratory; the Chowdhry section)

In [ ]:
adv = "Q2_harmful_content_medical_legal_financial_or_relationship_advice"
adv_rate = df.groupby("item_id")[adv].apply(lambda s: (s == "Yes").mean())
adv_items = adv_rate.index[adv_rate >= .2]
print(f"advice-harm items: {len(adv_items)}")
for col in GUARD_COLS:
    a_rows, _ = guard_table(col, race, subset=adv_items)
    o_rows, _ = guard_table(col, race, subset=race.index.difference(adv_items))
    ga, _ = gaps(a_rows); go, _ = gaps(o_rows)
    print(f"{col}: AUC gap advice={ga:.3f} (n={len(adv_items)}) vs other={go:.3f}")
print("Report as exploratory. Ns in every cell of the appendix table.")

## Sensitivity coding + LaTeX export for Table 1

In [ ]:
pooled_s, entropy_s, MAJ_s = build("unsafe_sens")
for col in GUARD_COLS:
    rows, _ = guard_table(col, MAJ_s[PRIMARY_AXIS])
    ag, kg = gaps(rows)
    print(f"[Unsure->unsafe] {col}: AUC gap={ag:.3f} kappa gap={kg:.3f}")

print("\n% ---- Table 1 rows (paste into whose_safety.tex) ----")
for grp in race.columns:
    cells_ = " & ".join(f"{report[c]['full'][grp[:12]][0]:.3f}" for c in GUARD_COLS)
    print(f"{grp[:12]} & {cells_} \\\\")
print("\\midrule")
print("AUC gap & " + " & ".join(f"{report[c]['auc_gap_full']:.3f}" for c in GUARD_COLS) + " \\\\")
print("95\\% CI & " + " & ".join(f"[{report[c]['auc_gap_ci95'][0]:.2f}, {report[c]['auc_gap_ci95'][1]:.2f}]" for c in GUARD_COLS) + " \\\\")
json.dump(report, open("analysis_report.json","w"), indent=1)
print("\nsaved analysis_report.json")